# HTMX SSE Integration Example

> Real-time progress tracking with HTMX using Server-Sent Events (SSE) for efficient streaming updates. Demonstrates server-side HTML streaming with automatic SSE connection cleanup using OOB (Out-of-Band) swaps, eliminating unnecessary connections while maintaining responsive UI updates.

In [1]:
from fasthtml.common import *
import uuid, time
import asyncio
import random

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream_async

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_sizes
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow, display_tw
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.CUPCAKE)
app.hdrs.append(Link(rel='icon', type='image/svg+xml', href=f'https://api.dicebear.com/9.x/adventurer/svg?seed={random.random()}'))

# Initialize
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Add HTMX SSE extension after HTMX script
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

htmx_idx = get_htmx_idx(app.hdrs)
if htmx_idx >= 0:
    # Insert SSE extension right after HTMX
    app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse"))
else:
    print("HTMX not found")

In [5]:
# Sample tasks
def file_processing_task():
    from tqdm import tqdm
    import time
    
    files = [f"file_{i}.txt" for i in range(50)]
    processed = []
    
    for file in tqdm(files, desc="Processing files"):
        time.sleep(0.05)  # Simulate file processing
        processed.append(file)
    
    return processed

def data_analysis_task():
    from tqdm import tqdm
    import time
    
    # Load data
    for _ in tqdm(range(30), desc="Loading data"):
        time.sleep(0.2)
    
    # Analyze
    for _ in tqdm(range(50), desc="Analyzing"):
        time.sleep(0.3)
    
    # Generate report
    for _ in tqdm(range(20), desc="Generating report"):
        time.sleep(0.2)
    
    return "Analysis complete"

In [6]:
# SSE helper functions
def render_task_buttons(disabled=False, oob_swap=False):
    """Render task buttons with appropriate state
    
    Args:
        disabled: Whether buttons should be disabled
        oob_swap: Whether to enable out-of-band swap for HTMX
    """
    btn_classes_files = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else "",
        m.r(2)
    )
    btn_classes_analysis = combine_classes(
        btn,
        btn_colors.secondary,
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'id': 'task-buttons',
        'cls': str(m.b(4))
    }
    
    if oob_swap:
        kwargs['hx_swap_oob'] = "true"
    
    return Div(
        Button(
            "Process Files",
            hx_post="/start_task" if not disabled else None,
            hx_vals='{"task_type": "files"}' if not disabled else None,
            hx_target="#progress-container" if not disabled else None,
            hx_swap="innerHTML" if not disabled else None,
            cls=btn_classes_files,
            disabled=disabled,
            id="process-files-btn"
        ),
        Button(
            "Run Analysis",
            hx_post="/start_task" if not disabled else None,
            hx_vals='{"task_type": "analysis"}' if not disabled else None,
            hx_target="#progress-container" if not disabled else None,
            hx_swap="innerHTML" if not disabled else None,
            cls=btn_classes_analysis,
            disabled=disabled,
            id="run-analysis-btn"
        ),
        **kwargs
    )

In [7]:
def create_progress_bar(value="0", max="100", color=None, width=None):
    """Create a styled progress bar with consistent styling"""
    classes = [progress]
    if color:
        classes.append(color)
    if width:
        classes.append(width)
    
    return Progress(
        value=str(value),
        max=str(max),
        cls=combine_classes(*classes)
    )

def create_status_badge(text, color):
    """Create a status badge"""
    return Span(
        text,
        cls=combine_classes(badge, color)
    )

In [8]:
def create_sse_progress_display(job_id, task_name):
    """Create a progress display connected to SSE stream"""
    return Div(
        H3(f"{task_name} Progress", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
        
        # Status alert
        Div(
            f"Job ID: {job_id[:8]}...",
            id=f"job-status-{job_id}",
            cls=combine_classes(alert, alert_colors.info, m.b(4))
        ),
        
        # Progress section with SSE connection
        Div(
            # Initial state
            P("Overall: 0%", cls=str(font_weight.bold), id="overall-text"),
            create_progress_bar(value="0", color=progress_colors.primary, width=w.full),
            Div(id="task-bars"),  # Container for task progress bars
            P("Starting...", cls=str(font_size.sm), id="status-text"),
            
            # HTMX SSE attributes
            hx_ext="sse",
            sse_connect=f"/stream_progress?job_id={job_id}",
            sse_swap="message",
            hx_swap="innerHTML",
            id=f"progress-{job_id}"
        ),
        
        cls=combine_classes(card, bg_dui.base_200, p(6))
    )

In [9]:
def create_final_response(job_id, task_name, progress_value=100, status_text="Completed!", 
                         bars_html=None, enable_buttons=True):
    """Create a final response with static display and OOB updates"""
    bars_content = bars_html if bars_html else []
    
    # Progress display with OOB swap to replace SSE-connected div
    progress_display = Div(
        P(f"Overall: {progress_value:.1f}%", cls=str(font_weight.bold)),
        create_progress_bar(value=str(int(progress_value)), color=progress_colors.primary, width=w.full),
        Div(*bars_content, id="task-bars"),
        P(status_text, cls=str(font_size.sm)),
        id=f"progress-{job_id}",  # Must match the SSE-connected div ID
        hx_swap_oob="true"  # Replace the SSE-connected div
    )
    
    # Status alert with OOB swap
    status_alert = Div(
        "Task completed!",
        id=f"job-status-{job_id}",
        cls=combine_classes(alert, alert_colors.success, m.b(4)),
        hx_swap_oob="true"
    )
    
    # Build response
    response_parts = [progress_display, status_alert]
    if enable_buttons:
        response_parts.append(render_task_buttons(disabled=False, oob_swap=True))
    
    return Div(*response_parts)

In [10]:
def render_job_row(job_id, job):
    """Render a complete job row with SSE streaming for active jobs"""
    is_running = not job['completed']
    
    if is_running:
        # Active job - use SSE for real-time updates
        return Tr(
            Td(job_id[:8] + "...", id=f"job-id-cell-{job_id}"),
            Td(
                Span(
                    f"{job['overall_progress']:.1f}%",
                    id=f"progress-span-{job_id}",
                    hx_ext="sse",
                    sse_connect=f"/stream_job_cell?job_id={job_id}&cell_type=progress",
                    sse_swap="message"
                ),
                id=f"job-progress-cell-{job_id}"
            ),
            Td(
                Span(
                    create_status_badge("Running", badge_colors.warning),
                    id=f"status-span-{job_id}",
                    hx_ext="sse",
                    sse_connect=f"/stream_job_cell?job_id={job_id}&cell_type=status",
                    sse_swap="message"
                ),
                id=f"job-status-cell-{job_id}"
            ),
            Td(
                Span(
                    str(len(job['bars'])),
                    id=f"bars-span-{job_id}",
                    hx_ext="sse",
                    sse_connect=f"/stream_job_cell?job_id={job_id}&cell_type=bars",
                    sse_swap="message"
                ),
                id=f"job-bars-cell-{job_id}"
            ),
            id=f"job-row-{job_id}"
        )
    else:
        # Completed job - static display
        return Tr(
            Td(job_id[:8] + "...", id=f"job-id-cell-{job_id}"),
            Td(f"{job['overall_progress']:.1f}%", id=f"job-progress-cell-{job_id}"),
            Td(create_status_badge("Complete", badge_colors.success), id=f"job-status-cell-{job_id}"),
            Td(str(len(job['bars'])), id=f"job-bars-cell-{job_id}"),
            id=f"job-row-{job_id}"
        )

In [11]:
@rt("/")
def index():
    return create_test_page(
        "HTMX SSE Progress Demo",
        Div(
            H1("HTMX + SSE Progress Tracking", cls=combine_classes(font_size._2xl, font_weight.bold, m.b(6))),
            
            # Task selection
            Div(
                H2("Select Task", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                render_task_buttons(disabled=False),  # Start with enabled buttons
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress container
            Div(id="progress-container", cls=str(m.b(6))),
            
            # Active jobs with SSE streaming
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                Div(
                    id="jobs-container",
                    hx_get="/jobs_table",
                    hx_trigger="load",
                    hx_swap="innerHTML",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [12]:
@rt("/start_task")
async def start_task(request):
    form = await request.form()
    task_type = form.get('task_type', 'files')
    
    job_id = str(uuid.uuid4())
    
    # Select task based on type
    if task_type == 'analysis':
        task = data_analysis_task
        task_name = "Data Analysis"
    else:
        task = file_processing_task
        task_name = "File Processing"
    
    # Start the job
    runner.start(
        job_id,
        task,
        patch_kwargs={
            "min_delta_pct": 2,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Small delay to allow monitor to register the job
    time.sleep(0.05)
    
    # Get updated jobs table for OOB swap
    jobs_table_update = jobs_table()
    
    # Return response with OOB swaps
    return Div(
        # Progress display with SSE connection
        create_sse_progress_display(job_id, task_name),
        
        # OOB swaps
        render_task_buttons(disabled=True, oob_swap=True),
        Div(
            jobs_table_update,
            id="jobs-container",
            hx_swap_oob="true",
            cls=str(overflow.x.auto)
        )
    )

In [13]:
@rt("/stream_progress")
def stream_progress(job_id: str):
    """SSE endpoint that streams progress updates as HTML"""
    
    async def html_stream():
        """Generate HTML updates for HTMX SSE"""
        import json
        
        try:
            # Use the async SSE stream generator
            async for data in sse_stream_async(
                monitor, 
                job_id, 
                interval=0.1,
                heartbeat=30.0,
                wait_for_start=True,
                start_timeout=5.0
            ):
                
                # Parse the SSE data format
                if data.startswith('data: '):
                    try:
                        # Extract JSON from SSE data line
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            
                            # Create HTML content based on progress
                            if progress_data.get('completed'):
                                # Job completed
                                # Check if any other jobs are running
                                all_jobs = monitor.all()
                                has_running = any(not job['completed'] for job in all_jobs.values())
                                
                                # Build final bars HTML
                                bars_html = []
                                if progress_data.get('bars'):
                                    for bar_info in progress_data['bars'].values():
                                        bars_html.append(
                                            Div(
                                                P(f"{bar_info['desc']}: 100% ({bar_info['tot']}/{bar_info['tot']})", 
                                                  cls=str(font_size.sm)),
                                                create_progress_bar(value="100", 
                                                                  color=progress_colors.accent, 
                                                                  width=w.full),
                                                cls=str(m.b(2))
                                            )
                                        )
                                
                                # Use helper to create final response
                                yield sse_message(create_final_response(
                                    job_id=job_id,
                                    task_name="Task",  # We don't track task name here
                                    progress_value=100,
                                    status_text="Task completed!",
                                    bars_html=bars_html,
                                    enable_buttons=not has_running
                                ))
                                break  # Exit cleanly after sending completion
                            else:
                                # Progress update
                                progress_value = progress_data.get('progress', 0)
                                
                                # Build sub-progress bars
                                bars_html = []
                                status_text = f"Processing... {progress_value:.1f}%"
                                
                                if progress_data.get('bars'):
                                    for bar_info in progress_data['bars'].values():
                                        bars_html.append(
                                            Div(
                                                P(f"{bar_info['desc']}: {bar_info['pct']:.1f}% ({bar_info['cur']}/{bar_info['tot']})", 
                                                  cls=str(font_size.sm)),
                                                create_progress_bar(value=str(int(bar_info['pct'])), 
                                                                  color=progress_colors.accent, 
                                                                  width=w.full),
                                                cls=str(m.b(2))
                                            )
                                        )
                                    status_text = f"Running... ({len(progress_data['bars'])} active bars)"
                                
                                html_content = Div(
                                    P(f"Overall: {progress_value:.1f}%", cls=str(font_weight.bold)),
                                    create_progress_bar(value=str(int(progress_value)), 
                                                      color=progress_colors.primary, 
                                                      width=w.full),
                                    Div(*bars_html, id="task-bars"),
                                    P(status_text, cls=str(font_size.sm))
                                )
                                
                                yield sse_message(html_content)
                    except json.JSONDecodeError:
                        pass  # Skip invalid data
                elif data.startswith('event: end'):
                    # Job not found or timed out
                    yield sse_message(create_final_response(
                        job_id=job_id,
                        task_name="Task",
                        progress_value=0,
                        status_text="Job not found or timed out",
                        enable_buttons=True
                    ))
                    break
                elif data.startswith(': '):
                    # Heartbeat/keep-alive
                    yield data
                    
        except Exception as e:
            # Error - send final response
            yield sse_message(create_final_response(
                job_id=job_id,
                task_name="Task",
                progress_value=0,
                status_text=f"Stream error: {str(e)}",
                enable_buttons=True
            ))
    
    return EventStream(html_stream())

In [14]:
@rt("/stream_job_cell")
def stream_job_cell(job_id: str, cell_type: str):
    """SSE endpoint for streaming individual table cell updates"""
    
    async def cell_stream():
        """Generate cell updates for HTMX SSE"""
        import json
        
        try:
            # Check if job exists and is still running before starting stream
            snapshot = monitor.snapshot(job_id)
            if not snapshot or snapshot['completed']:
                # Job doesn't exist or already completed - send static span immediately
                if cell_type == "progress":
                    value = f"{snapshot['overall_progress']:.1f}%" if snapshot else "N/A"
                    yield sse_message(
                        Span(
                            value,
                            id=f"progress-span-{job_id}",
                            hx_swap_oob="true"
                        )
                    )
                elif cell_type == "status":
                    status = create_status_badge("Complete", badge_colors.success) if snapshot and snapshot['completed'] else "N/A"
                    yield sse_message(
                        Span(
                            status,
                            id=f"status-span-{job_id}",
                            hx_swap_oob="true"
                        )
                    )
                elif cell_type == "bars":
                    value = str(len(snapshot['bars'])) if snapshot else "0"
                    yield sse_message(
                        Span(
                            value,
                            id=f"bars-span-{job_id}",
                            hx_swap_oob="true"
                        )
                    )
                return  # Exit early without creating stream
            
            async for data in sse_stream_async(
                monitor, 
                job_id, 
                interval=0.5,  # Less frequent updates for table cells
                heartbeat=30.0,
                wait_for_start=False,  # Don't wait since we already checked
                start_timeout=5.0
            ):
                
                if data.startswith('data: '):
                    try:
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            
                            if progress_data.get('completed'):
                                # Job completed - send final update and close SSE
                                if cell_type == "progress":
                                    yield sse_message(
                                        Span(
                                            "100.0%",
                                            id=f"progress-span-{job_id}",
                                            hx_swap_oob="true"
                                        )
                                    )
                                elif cell_type == "status":
                                    # Check if we need to update button state
                                    all_jobs = monitor.all()
                                    has_running = any(not job['completed'] for job in all_jobs.values())
                                    
                                    if not has_running:
                                        yield sse_message(
                                            Div(
                                                Span(
                                                    create_status_badge("Complete", badge_colors.success),
                                                    id=f"status-span-{job_id}",
                                                    hx_swap_oob="true"
                                                ),
                                                render_task_buttons(disabled=False, oob_swap=True)
                                            )
                                        )
                                    else:
                                        yield sse_message(
                                            Span(
                                                create_status_badge("Complete", badge_colors.success),
                                                id=f"status-span-{job_id}",
                                                hx_swap_oob="true"
                                            )
                                        )
                                elif cell_type == "bars":
                                    # Send final bars count
                                    bars_count = str(len(progress_data.get('bars', {})))
                                    yield sse_message(
                                        Span(
                                            bars_count,
                                            id=f"bars-span-{job_id}",
                                            hx_swap_oob="true"
                                        )
                                    )
                                break
                            else:
                                if cell_type == "progress":
                                    progress_value = progress_data.get('progress', 0)
                                    yield sse_message(f"{progress_value:.1f}%")
                                elif cell_type == "status":
                                    yield sse_message(create_status_badge("Running", badge_colors.warning))
                                elif cell_type == "bars":
                                    # Update bars count
                                    bars_count = str(len(progress_data.get('bars', {})))
                                    yield sse_message(bars_count)
                    except json.JSONDecodeError:
                        pass
                elif data.startswith('event: end'):
                    # Job not found - close SSE connection with static span
                    if cell_type == "progress":
                        yield sse_message(
                            Span(
                                "N/A",
                                id=f"progress-span-{job_id}",
                                hx_swap_oob="true"
                            )
                        )
                    elif cell_type == "status":
                        yield sse_message(
                            Span(
                                create_status_badge("Error", badge_colors.error),
                                id=f"status-span-{job_id}",
                                hx_swap_oob="true"
                            )
                        )
                    elif cell_type == "bars":
                        yield sse_message(
                            Span(
                                "0",
                                id=f"bars-span-{job_id}",
                                hx_swap_oob="true"
                            )
                        )
                    break
                elif data.startswith(': '):
                    # Heartbeat
                    yield data
                    
        except Exception as e:
            # On error, close SSE connection with static span
            if cell_type == "progress":
                yield sse_message(
                    Span(
                        "Error",
                        id=f"progress-span-{job_id}",
                        hx_swap_oob="true"
                    )
                )
            elif cell_type == "status":
                yield sse_message(
                    Span(
                        create_status_badge("Error", badge_colors.error),
                        id=f"status-span-{job_id}",
                        hx_swap_oob="true"
                    )
                )
            elif cell_type == "bars":
                yield sse_message(
                    Span(
                        "0",
                        id=f"bars-span-{job_id}",
                        hx_swap_oob="true"
                    )
                )
    
    return EventStream(cell_stream())

In [15]:
@rt("/jobs_table")
def jobs_table():
    """Jobs table with SSE streaming for active jobs"""
    jobs = monitor.all()
    
    if not jobs:
        return P("No active jobs", cls=str(text_color.gray(500)))
    
    # Create table with jobs using SSE for active jobs
    rows = []
    for job_id, job in jobs.items():
        rows.append(render_job_row(job_id, job))
    
    return Table(
        Thead(
            Tr(
                Th("Job ID"),
                Th("Progress"),
                Th("Status"),
                Th("Bars")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_sizes.xs, w.full)
    )

In [16]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [17]:
server.stop()